(course-enzyme-physicochemical-properties)=

# Path B - Module 39: Physicochemical Properties

To finalize your industrial design, you need scientific numbers that quantify the "pocket" of your enzyme. How much surface area is the plastic substrate losing when it enters the catalytic cleft? 

In this module, you will learn to use **Buried Area** and other physical descriptors to prove the efficiency of your engineered PETase.

In [ ]:
import molsysmt as msm
from molsysmt import systems

# Load our solvated complex
petase = msm.convert('pdb:6EQE', to_form='molsysmt.MolSys', selection='molecule_type=="protein"')
bhet = msm.convert('C1=CC(=CC=C1C(=O)OCCO)C(=O)OCCO', from_form='string:smiles', to_form='molsysmt.MolSys')
molsys = msm.merge([petase, bhet])

(course-enzyme-physicochemical-properties-aromatic-participants)=
### Aromatic participants

`msm.topology.get_rings()` describes a minimum cycle basis of the selected
chemical state's covalent graph. `msm.physchem.get_aromatic_rings()` prepares
participants from the explicitly aromatic bond subgraph. Aromaticity belongs to
ChemicalStates, not to the topology inventory or geometric planarity. Missing
aromatic flags are unknown, not False. A covalent cycle can be nonaromatic.

Both tools accept supported forms providing the required chemistry, including
typed ChemicalStatesDict and chemistry-only MolSys. Missing aromatic metadata
is reported explicitly.

Both tools return sparse, complete atom-index memberships and preserve the source
state. Tied bases need not preserve molecular symmetry; nonaromatic fusion bonds
can leave an aromatic perimeter cycle. A partial fused-ring selection can cut
another ring sharing atoms and must be expanded. The cyclic-block limit bounds
individual basis problems. These experimental chemical tools do not yet calculate
pi–pi observations. See {ref}`Tutorial_Get_rings`,
{ref}`Tutorial_Get_aromatic_rings`, and
{ref}`Cookbook_Preparing_aromatic_participants` for executed examples.

The general `msm.structure.get_least_squares_plane()` tool fits unweighted planes to these
complete memberships, reporting centroid, unoriented normal and orthogonal RMS/
maximum deviations. It needs coordinates only and therefore also supports
Structures-only systems. Collinear or degenerate clouds fail explicitly;
planarity does not infer aromaticity. PBC validation requires groups already
whole in one minimum-image neighborhood. Input streaming does not remove the
RAM cost of the dense result. See {ref}`Tutorial_Get_least_squares_plane` and the executed
{ref}`Cookbook_Preparing_aromatic_participants` recipe.


### 1. Global Charge and Mass
Check if your His-tag and your mutations changed the total charge of the enzyme, which could affect its solubility in the bioreactor.

In [ ]:
total_charge = msm.physchem.get_charge(molsys, selection='molecule_type=="protein"')
total_mass = msm.physchem.get_mass(molsys, selection='molecule_type=="protein"')

print(f"Engineered Enzyme Charge: {total_charge}")
print(f"Engineered Enzyme Mass: {total_mass}")

### 2. Interface Analysis: Buried Area
A good biocatalyst should have a high buried area for its substrate, meaning the substrate is well-protected and oriented within the active site.

In [ ]:
# Calculate the surface area that is buried upon complexation
buried = msm.physchem.get_area_buried(molsys, selection='molecule_type=="protein"', 
                                      selection_2='molecule_type=="small molecule"')

print(f"Lost surface area (Buried Area): {buried}")

# Calculate the fraction of the plastic monomer that is hidden inside the enzyme
fraction = msm.physchem.get_buried_fraction(molsys, selection='molecule_type=="small molecule"', 
                                            selection_2='molecule_type=="protein"')

print(f"Percentage of BHET inside the pocket: {fraction*100:.1f}%")

(course-enzyme-physicochemical-properties-charge-centers)=
### Charge centers

Formal charges belong to a selected ChemicalStates state; force-field partial
charges belong to MolecularMechanics. A residue descriptor returned by
msm.physchem.get_charge() is a different definition and cannot establish
atom-level ionic-center membership.

The experimental msm.physchem.get_charge_centers() operation prepares sparse
chemical features with whole-participant membership, geometric reference atoms,
charges in elementary charge units, source indices, and scientific provenance.
It groups bounded carboxyl/guanidine motifs and directly connected charged atoms,
and omits groups of zero net charge. A zwitterion can therefore have two local
centers while its total charge is zero. Other charged atoms remain labeled
atomic centers; phosphate, sulfate and aromatic delocalization are not grouped.

Required chemistry must be present and connectivity declared complete; the
explicit assume_complete_connectivity option records a caller assumption and
never reconstructs missing chemistry. See {ref}`Tutorial_Get_charge_centers`
and {ref}`Cookbook_Preparing_charged_participants` for executed examples.
These chemical features do not themselves represent observed contacts or
electrostatic energies. You can now use
`msm.interactions.ionic.get_ionic_interactions()` with an explicit unitful
minimum-distance cutoff to calculate a sparse analysis. It supports internal,
incident and disjoint-between searches; queries retain whole participants and
evaluated-empty structures. Attach the analysis under a name explicitly.
The route supports eager and chunked calculations. Chunked native MolSys
and H5MSM 0.5 inputs require atom-index selections or all, and prepare
chemistry once. The complete sparse result still requires RAM. Periodic
compound groups requiring internal image shifts remain unsupported. See {ref}`Tutorial_Get_ionic_interactions` and the
{ref}`Cookbook_Saving_ionic_interactions` recipe.

--- 

### 🏁 END OF PHASE 4: DATA ANALYST

**Congratulations!** You have completed the biophysical characterization of your industrial enzyme. You have:
1.  **Measured** catalytic distances.
2.  **Mapped** the binding pocket neighbors.
3.  **Visualized** interaction matrices.
4.  **Assessed** stability via RG and PCA.
5.  **Verified** the secondary structure fold.
6.  **Quantified** H-bond networks and Buried Surface Area.

Now that your characterization is solid, we move to **Phase 5: The Physics Lab**, where we will simulate the enzyme's behavior under industrial conditions.

(course-enzyme-physicochemical-properties-aromatic-observations)=
### Aromatic observations

After preparing declared aromatic participants and their fitted planes, use
`msm.interactions.pi_pi.get_pi_pi_interactions()` with explicit distance, angular,
lateral-offset and planarity cutoffs. Parallel and edge-to-face geometries are
experimental observations, not energy calculations. Self, overlapping/fused and
directly covalently linked rings are excluded. Calculation selections must contain
whole rings; completed analyses support queries by individual atoms and frames.

Each evaluated frame, including empty ones, remains explicit. Bounded native or
H5MSM coordinate blocks feed a resident sparse result. Periodic images shift the
second whole ring relative to the first; split groups fail. Attach the result under
a name in `molsys.interactions`, then preserve it through public H5MSM conversion.
See the executed {ref}`Tutorial_Get_pi_pi_interactions` and
{ref}`Cookbook_Saving_pi_pi_interactions` for reproducible examples.


(course-enzyme-physicochemical-properties-cation-pi)=
### Attributed cation–π observations

`msm.interactions.cation_pi.get_cation_pi_interactions()` defaults to ProLIF 2.2.2's
chemical SMARTS and original centroid-edge normal, with a 0.45 nm/0–30° criterion.
`msm.topology.get_substructure_matches()` supplies reusable, bounded chemical matching
on the full declared state before selection. It preserves query atom order and does
not truncate silently. Bracket-declared hydrogens remain ChemicalStates annotations,
separate from implicit counts or real indexed H atoms.

The separately named `centroid_angle_offset` method uses formal-charge groups,
whole-group centroids and fitted planes, with four explicit cutoffs. It is a proposal
under comparison, not a reproduction of ProLIF or an established improvement. Both
return sparse geometric evidence and evaluated empty frames, with optional MIC and
explicit named attachment. Source producer versions and original method references
survive H5MSM. See the executed {ref}`Tutorial_Get_cation_pi_interactions`,
{ref}`Tutorial_Get_substructure_matches` and
{ref}`Cookbook_Saving_cation_pi_interactions` examples.
